# Finite-temperature phonons with pheasy and machine-learned potentials

This notebook runs `ForceFieldFiniteTemperaturePhononMaker` with three MACE
potentials. The workflow fits effective harmonic force constants to MD
snapshots at a given temperature. The workflow is new and less tested than the
other phonon workflows, so its defaults may still change.

## Contents

1. Five materials from 100 to 900 K
2. Thermal expansion with an NPT MD
3. MD length and number of snapshots
4. Li3PS4, a crystal that is unstable at 0 K
5. Li3PS4 with hopping Li: diffusion, DOS and free energies from the MD
6. Known limitations

The long runs are switched off. Our results are shown as figures and tables.

## How the workflow works

1. Relax the structure (optional).
2. Build a diagonal supercell with lattice vectors of at least `min_length`,
   12 Å by default.
3. Optional: run an NPT MD to get the cell at the temperature (Section 2).
4. Run an NVT MD, 8 ps with a 1 fs time step by default.
5. Leave out the first 1 ps and pick 50 evenly spaced snapshots.
6. Compute the forces on the snapshots and on the undisplaced supercell.
7. Fit the second-order force constants with pheasy (LASSO).
8. Compute the band structure and the DOS. Imaginary modes are reported, not
   removed.

A check of the trajectory flags melting, a shift away from the starting
structure and a drift of the energy. It raises a warning, and the fit is still
done. The idea is that of TDEP (Hellman et al., Phys. Rev. B 84, 180301
(2011)). The force constants include the effect of the anharmonic forces on the
frequencies. They give no lifetimes, and the MD is classical. The VASP version
is `atomate2.vasp.flows.finite_temperature_phonons.FiniteTemperaturePhononMaker`.

## Thermostat

The NVT MD uses a Langevin thermostat with a friction of 10 ps⁻¹. It samples
the canonical ensemble for every mode (Bussi and Parrinello, Phys. Rev. E 75,
056707 (2007)). `thermostat="nose-hoover"` is also available, but a weakly
coupled Nose-Hoover thermostat is not ergodic for a harmonic oscillator (Legoll
et al., Arch. Ration. Mech. Anal. 184, 449 (2007)). The NPT MD uses ASE's
`MTKNPT`. `random_seed` seeds the initial velocities and the Langevin forces.

## Installation

```
pip install 'atomate2[pheasy,ase]'
pip install 'mace-torch>=0.3.16' mp-api
```

The `pheasy` extra builds ALM from source. The hiPhive tutorial lists what the
build needs.

## Potentials

- MACE-OMAT-0-medium. `mace_mp(model="medium-omat-0")` downloads it.
- MACE-MATPES-PBE-0 and MACE-MATPES-r2SCAN-0. Download
  [MACE-matpes-pbe-omat-ft.model](https://github.com/ACEsuit/mace-foundations/releases/download/mace_matpes_0/MACE-matpes-pbe-omat-ft.model)
  and
  [MACE-matpes-r2scan-omat-ft.model](https://github.com/ACEsuit/mace-foundations/releases/download/mace_matpes_0/MACE-matpes-r2scan-omat-ft.model)
  and set their paths in the next cell.

All three use the Academic Software License. Section 1 only needs
MACE-OMAT-0-medium. We ran in `float64` on one NVIDIA A100 GPU.

In [ ]:
import os
import warnings

# macOS: conda's llvm-openmp and torch's bundled libomp both load and the
# duplicate aborts the process. This must be set before torch is imported.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
# hide the warnings of torch, e3nn and MACE when the models are loaded
warnings.filterwarnings("ignore", module=r"(torch|e3nn|mace)(\.|$)")

import torch  # noqa: E402
from mace.calculators import mace_mp  # noqa: E402

# Replace the two paths with the files you downloaded.
MODEL_FILES = {
    "OMAT-0-medium": "medium-omat-0",
    "MATPES-PBE-0": "/path/to/MACE-matpes-pbe-omat-ft.model",
    "MATPES-r2SCAN-0": "/path/to/MACE-matpes-r2scan-omat-ft.model",
}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def calc_kwargs(model: str) -> dict:
    """Return the calculator settings for one of the models above."""
    return {"model": MODEL_FILES[model], "device": DEVICE, "default_dtype": "float64"}


_ = mace_mp(**calc_kwargs("OMAT-0-medium"))  # downloads and caches on first use

## One calculator for all jobs

`run_locally` runs all jobs in one process, and each force field job loads its
own calculator. The next cell makes them share one, as in the thermal expansion
tutorial. A workflow manager does not need this.

In [ ]:
from ase.calculators.calculator import Calculator

import atomate2.forcefields.utils as ff_utils

_ase_calculator = ff_utils.ase_calculator
_calculators: dict[str, Calculator] = {}


def shared_ase_calculator(calculator_meta: object, **kwargs: object) -> Calculator:
    """Build each calculator once and reuse it for every job."""
    key = repr((calculator_meta, sorted(kwargs.items())))
    if key not in _calculators:
        _calculators[key] = _ase_calculator(calculator_meta, **kwargs)
    _calculators[key].reset()
    return _calculators[key]


ff_utils.ase_calculator = shared_ase_calculator

## 1. Five materials from 100 to 900 K

We first run Ca3Ir4Sn13 (mp-1200211) at 300 K with MACE-OMAT-0-medium. It has
40 atoms in the primitive cell and imaginary modes at 0 K. We take its PBEsol
cell from the Materials Project's Harmonic Phonon Database. Set
`MP_API_KEY` first.

In [ ]:
from mp_api.client import MPRester
from pymatgen.core import Structure

if not os.environ.get("MP_API_KEY"):
    raise OSError("export MP_API_KEY before running this cell")


def get_pheasy_structure(mp_id: str) -> Structure:
    """Get the PBEsol cell of a material in the Harmonic Phonon Database."""
    with MPRester() as mpr:
        (doc,) = mpr.materials.phonon.search(
            material_ids=[mp_id],
            phonon_method="pheasy",
            fields=["structure"],
            num_chunks=1,
            chunk_size=10,
        )
    return doc.structure


structure = get_pheasy_structure("mp-1200211")
structure.lattice

## Building the workflow

`from_force_field_name` uses one force field for every job. "MACE-MP-0" picks
the MACE calculator, and `calculator_kwargs` picks the model. The cell is
already relaxed with PBEsol, so we skip the relaxation. We use the 2x2x2
supercell (320 atoms) of our AIMD reference.

In [ ]:
from atomate2.forcefields.flows.finite_temperature_phonons import (
    ForceFieldFiniteTemperaturePhononMaker,
)

maker = ForceFieldFiniteTemperaturePhononMaker.from_force_field_name(
    "MACE-MP-0",
    calculator_kwargs=calc_kwargs("OMAT-0-medium"),
    relax_initial_structure=False,
    temperature=300,
)
flow = maker.make(structure, supercell_matrix=[[2, 0, 0], [0, 2, 0], [0, 0, 2]])
flow.draw_graph().show()

## Running the workflow

The flow has one MD job and 51 force calculations. It took 12 minutes on one
A100 GPU. `create_folders=True` is needed, since the snapshots are read from the
trajectory file of the MD job.

In [ ]:
from jobflow import JobStore, run_locally
from maggma.stores import MemoryStore

job_store = JobStore(MemoryStore(), additional_stores={"data": MemoryStore()})
responses = run_locally(flow, store=job_store, create_folders=True, ensure_success=True)
doc = responses[flow.output.uuid][1].output

The output is a `FiniteTemperaturePhononDoc`.

In [ ]:
import numpy as np

health = doc.trajectory_health
{
    "highest frequency in THz": float(np.max(doc.phonon_bandstructure.bands)),
    "imaginary modes on the band path": doc.has_imaginary_modes,
    "lowest frequency at the commensurate q-points in THz": doc.lowest_frequency,
    "LASSO penalty": doc.lasso_alpha,
    "force RMSE in eV/A": doc.force_rmse,
    "trajectory check": health.verdict,
    "Lindemann ratio": health.lindemann_ratio,
}

Our run gave a highest frequency of 5.97 THz, no imaginary modes and a stable
trajectory. `has_imaginary_modes` checks the band path. `lowest_frequency` uses
only the q-points of the supercell, where the fit sets the frequencies directly
(Section 3). The check reports melting above a Lindemann ratio of 0.15, the
value at melting of fcc solids (Saija et al., J. Chem. Phys. 124, 244504
(2006)).

## Comparing with 0 K

The next cell plots our 300 K bands next to the 0 K PBEsol bands of the
Materials Project's Harmonic Phonon Database (Sahasrabuddhe et al., ChemRxiv
(2026), doi:10.26434/chemrxiv.15004632/v1).

In [ ]:
import re
from functools import cache

import matplotlib.pyplot as plt
from phonopy import Phonopy
from pymatgen.io.phonopy import get_phonopy_structure
from pymatgen.phonon.bandstructure import PhononBandStructureSymmLine
from pymatgen.phonon.dos import PhononDos


def plot_bands(
    ax: plt.Axes,
    bs: PhononBandStructureSymmLine,
    label: str | None = None,
    distance: list[float] | None = None,
    **style,
) -> None:
    """Plot a phonon band structure, one segment of the path at a time.

    distance replaces the distances along the path of bs, for a band structure
    computed on the same q-points as another one.
    """
    distance = np.array(bs.distance if distance is None else distance)
    positions, names = [], []
    for idx, branch in enumerate(bs.branches):
        part = slice(branch["start_index"], branch["end_index"] + 1)
        lines = ax.plot(distance[part], bs.bands[:, part].T, **style)
        if idx == 0 and label:
            lines[0].set_label(label)
        start, end = branch["name"].split("-")
        if positions and np.isclose(positions[-1], distance[part.start]):
            if names[-1] != start:
                names[-1] += "|" + start
        else:
            positions.append(distance[part.start])
            names.append(start)
        positions.append(distance[branch["end_index"]])
        names.append(end)
    names = [name.replace("\\Gamma", "Γ").replace("GAMMA", "Γ") for name in names]
    names = [re.sub(r"_(\w+)", r"$_{\1}$", name) for name in names]
    ax.set_xticks(positions, names)
    ax.set_xlim(distance[0], distance[-1])
    ax.axhline(0, color="gray", lw=0.5)


def plot_dos(ax: plt.Axes, dos: PhononDos, label: str | None = None, **style) -> None:
    """Plot a phonon DOS sideways, to sit next to a band structure."""
    ax.plot(dos.densities, dos.frequencies, label=label, **style)
    ax.set_xlabel("DOS")


@cache
def get_pheasy_phonons(mp_id: str) -> tuple[PhononBandStructureSymmLine, PhononDos]:
    """Get the 0 K PBEsol band structure and DOS of the Harmonic Phonon Database.

    path_type="hinuma" gives the seekpath band path that the workflow uses.
    """
    with MPRester() as mpr:
        bs = mpr.materials.phonon.get_bandstructure_from_material_id(
            mp_id, phonon_method="pheasy", path_type="hinuma"
        ).to_pmg
        dos = mpr.materials.phonon.get_dos_from_material_id(
            mp_id, phonon_method="pheasy"
        ).to_pmg
    return bs, dos


def get_pheasy_bands(
    mp_id: str, bs: PhononBandStructureSymmLine
) -> PhononBandStructureSymmLine:
    """Get the 0 K PBEsol bands of the Harmonic Phonon Database on the q-points of bs.

    The stored bands are returned if they share the q-points of bs. Otherwise, as
    for Bi4S3N2, phonopy computes them from the stored force constants.
    """
    bs_0k, _ = get_pheasy_phonons(mp_id)
    qpoints = np.array([qpoint.frac_coords for qpoint in bs.qpoints])
    stored = np.array([qpoint.frac_coords for qpoint in bs_0k.qpoints])
    if stored.shape == qpoints.shape and np.allclose(stored, qpoints, atol=1e-4):
        return bs_0k
    with MPRester() as mpr:
        (doc,) = mpr.materials.phonon.search(
            material_ids=[mp_id],
            phonon_method="pheasy",
            fields=["structure", "supercell_matrix", "primitive_matrix"],
        )
        force_constants = mpr.materials.phonon.get_forceconstants_from_material_id(
            mp_id, phonon_method="pheasy"
        )
    phonon = Phonopy(
        get_phonopy_structure(doc.structure),
        supercell_matrix=doc.supercell_matrix,
        primitive_matrix=doc.primitive_matrix,
    )
    phonon.force_constants = np.array(force_constants)
    phonon.run_qpoints(qpoints)
    labels = {label: kpoint.frac_coords for label, kpoint in bs.labels_dict.items()}
    return PhononBandStructureSymmLine(
        qpoints, phonon.qpoints.frequencies.T, bs.lattice_rec, labels_dict=labels
    )


bs_0k, dos_0k = get_pheasy_phonons("mp-1200211")

fig, axes = plt.subplots(
    1, 2, figsize=(9, 4), sharey=True, gridspec_kw={"width_ratios": [3, 1]}
)
for bs, dos, label, color in (
    (bs_0k, dos_0k, "0 K, PBEsol", "black"),
    (doc.phonon_bandstructure, doc.phonon_dos, "300 K, MACE-OMAT-0-medium", "C0"),
):
    plot_bands(axes[0], bs, label=label, color=color, lw=0.6)
    plot_dos(axes[1], dos, color=color)
axes[0].set_ylabel("Frequency (THz)")
axes[0].legend(fontsize=8)
plt.show()

![Band structure and DOS of Ca3Ir4Sn13 at 0 K and at 300 K](finite_temperature_phonons_figures/s1_ca3ir4sn13_300k.png)

The imaginary modes of 0 K, down to -0.75 THz, are gone at 300 K. AIMD with
PBEsol in the same supercell gives a highest frequency of 6.03 THz. Over the
band path, MACE-OMAT-0-medium differs from it by 0.07 THz on average.

## All temperatures, potentials and materials

The loop runs five materials with three potentials at five temperatures. AlAsPt5,
Zn3P2 and Er5Tl3 also have imaginary modes at 0 K, and Bi4S3N2 is polar. The
loop is off by default. Each run took 6 to 22 minutes.

In [ ]:
from atomate2.common.schemas.finite_temperature_phonons import (
    FiniteTemperaturePhononDoc,
)

# mp-id and the diagonal of the supercell matrix of our AIMD reference
MATERIALS = {
    "Ca3Ir4Sn13": ("mp-1200211", (2, 2, 2)),
    "AlAsPt5": ("mp-1025306", (4, 4, 2)),
    "Zn3P2": ("mp-2071", (2, 2, 2)),
    "Bi4S3N2": ("mp-1245549", (2, 2, 2)),
    "Er5Tl3": ("mp-1105965", (2, 2, 2)),
    "KNaICl": ("mp-1002081", (3, 3, 2)),
}
TEMPERATURES = (100, 300, 500, 700, 900)


def run_ft(
    structure: Structure,
    supercell: tuple[int, int, int],
    model: str,
    root_dir: str,
    temperature: float,
    md_time: float = 8.0,
    n_snapshots: int = 50,
    relax: bool = False,
    run_npt: bool = False,
) -> FiniteTemperaturePhononDoc:
    """Run the finite-temperature phonon workflow and return its output."""
    maker = ForceFieldFiniteTemperaturePhononMaker.from_force_field_name(
        "MACE-MP-0",
        calculator_kwargs=calc_kwargs(model),
        relax_initial_structure=relax,
        run_npt=run_npt,
        temperature=temperature,
        md_time=md_time,
        n_snapshots=n_snapshots,
    )
    flow = maker.make(structure, supercell_matrix=np.diag(supercell).tolist())
    os.makedirs(root_dir, exist_ok=True)
    responses = run_locally(
        flow,
        store=JobStore(MemoryStore(), additional_stores={"data": MemoryStore()}),
        create_folders=True,
        root_dir=root_dir,
        ensure_success=True,
    )
    return responses[flow.output.uuid][1].output


RUN_ALL = False
docs = {}
if RUN_ALL:
    for name in ("Ca3Ir4Sn13", "AlAsPt5", "Zn3P2", "Bi4S3N2", "Er5Tl3"):
        mp_id, supercell = MATERIALS[name]
        unit_cell = get_pheasy_structure(mp_id)
        for model in MODEL_FILES:
            for temperature in TEMPERATURES:
                docs[name, model, temperature] = run_ft(
                    unit_cell,
                    supercell,
                    model,
                    f"runs/{name}_{model}_{temperature}K",
                    temperature,
                )

The next cell plots the bands and the DOS at 100, 500 and 900 K. The black lines
are the 0 K PBEsol phonons of the Materials Project's Harmonic Phonon Database.
The stored band path of Bi4S3N2 differs from ours, so its 0 K bands are computed
from the stored force constants.

In [ ]:
def plot_temperatures(name: str, label: str | None = None) -> None:
    """Plot the band structures and DOS of one material at 100, 500 and 900 K.

    The 0 K PBEsol phonons are drawn in black. label picks the runs of Section 2,
    "relaxed" or "npt". None picks Section 1.
    """
    suffix = () if label is None else (label,)
    fig, axes = plt.subplots(
        1,
        2 * len(MODEL_FILES),
        figsize=(5.5 * len(MODEL_FILES), 4),
        sharey=True,
        gridspec_kw={"width_ratios": [3, 1] * len(MODEL_FILES)},
    )
    colors = dict(
        zip(TEMPERATURES, plt.cm.plasma(np.linspace(0.15, 0.85, 5)), strict=True)
    )
    mp_id = MATERIALS[name][0]
    _, dos_0k = get_pheasy_phonons(mp_id)
    for idx, model in enumerate(MODEL_FILES):
        reference = docs[(name, model, 100, *suffix)].phonon_bandstructure
        bs_0k = get_pheasy_bands(mp_id, reference)
        style = {"label": "0 K, PBEsol", "color": "black"}
        plot_bands(axes[2 * idx], bs_0k, distance=reference.distance, lw=0.6, **style)
        plot_dos(axes[2 * idx + 1], dos_0k, **style)
        for temperature in (100, 500, 900):
            doc = docs[(name, model, temperature, *suffix)]
            style = {"label": f"{temperature} K", "color": colors[temperature]}
            plot_bands(axes[2 * idx], doc.phonon_bandstructure, lw=0.6, **style)
            plot_dos(axes[2 * idx + 1], doc.phonon_dos, **style)
        axes[2 * idx].set_title(model)
    axes[0].set_ylabel("Frequency (THz)")
    axes[-1].legend(fontsize=8)
    fig.suptitle(name if label is None else f"{name}, {label} cell", y=1.02)
    plt.show()


if RUN_ALL:
    for name in ("Ca3Ir4Sn13", "AlAsPt5", "Zn3P2", "Bi4S3N2", "Er5Tl3"):
        plot_temperatures(name)

The figures below show our runs.

![Band structures and DOS of Ca3Ir4Sn13 at 0 K with PBEsol and at 100, 500 and 900 K](finite_temperature_phonons_figures/s1_bands_Ca3Ir4Sn13.png)

![Band structures and DOS of AlAsPt5 at 0 K with PBEsol and at 100, 500 and 900 K](finite_temperature_phonons_figures/s1_bands_AlAsPt5.png)

![Band structures and DOS of Zn3P2 at 0 K with PBEsol and at 100, 500 and 900 K](finite_temperature_phonons_figures/s1_bands_Zn3P2.png)

![Band structures and DOS of Bi4S3N2 at 0 K with PBEsol and at 100, 500 and 900 K](finite_temperature_phonons_figures/s1_bands_Bi4S3N2.png)

![Band structures and DOS of Er5Tl3 at 0 K with PBEsol and at 100, 500 and 900 K](finite_temperature_phonons_figures/s1_bands_Er5Tl3.png)

## Our results

The dashed line is the highest frequency of AIMD at 300 K.

In [ ]:
import pandas as pd

# highest frequency on the band path in THz, at 100, 300, 500, 700 and 900 K
MAX_FREQUENCY = {
    "Ca3Ir4Sn13": {
        "OMAT-0-medium": (6.03, 5.97, 5.94, 5.89, 5.78),
        "MATPES-PBE-0": (5.80, 5.75, 5.74, 5.66, 5.64),
        "MATPES-r2SCAN-0": (6.03, 5.95, 5.91, 5.89, 5.78),
    },
    "AlAsPt5": {
        "OMAT-0-medium": (7.80, 7.76, 7.70, 7.77, 7.78),
        "MATPES-PBE-0": (7.58, 7.66, 7.68, 7.67, 7.80),
        "MATPES-r2SCAN-0": (7.65, 7.58, 7.54, 7.68, 7.58),
    },
    "Zn3P2": {
        "OMAT-0-medium": (10.73, 10.44, 10.05, 9.80, 9.42),
        "MATPES-PBE-0": (10.42, 10.08, 9.91, 9.63, 9.27),
        "MATPES-r2SCAN-0": (11.26, 10.91, 10.69, 10.33, 10.14),
    },
    "Bi4S3N2": {
        "OMAT-0-medium": (17.35, 16.93, 15.98, 15.85, 15.12),
        "MATPES-PBE-0": (15.64, 15.21, 15.28, 14.56, 14.53),
        "MATPES-r2SCAN-0": (17.60, 17.23, 16.52, 16.00, 15.73),
    },
    "Er5Tl3": {
        "OMAT-0-medium": (4.36, 4.25, 4.28, 4.24, 4.19),
        "MATPES-PBE-0": (4.15, 4.12, 4.05, 3.91, 3.86),
        "MATPES-r2SCAN-0": (4.18, 4.14, 4.15, 4.11, 3.98),
    },
}
# AIMD reference at 300 K: highest frequency in THz, and the mean absolute
# difference of each potential from it over all branches of the band path
AIMD_MAX_300K = {
    "Ca3Ir4Sn13": 6.03,
    "AlAsPt5": 7.79,
    "Zn3P2": 10.09,
    "Bi4S3N2": 17.03,
    "Er5Tl3": 3.97,
}
MAE_300K = {
    "Ca3Ir4Sn13": {
        "OMAT-0-medium": 0.07,
        "MATPES-PBE-0": 0.17,
        "MATPES-r2SCAN-0": 0.10,
    },
    "AlAsPt5": {"OMAT-0-medium": 0.11, "MATPES-PBE-0": 0.19, "MATPES-r2SCAN-0": 0.17},
    "Zn3P2": {"OMAT-0-medium": 0.21, "MATPES-PBE-0": 0.10, "MATPES-r2SCAN-0": 0.44},
    "Bi4S3N2": {"OMAT-0-medium": 0.15, "MATPES-PBE-0": 0.46, "MATPES-r2SCAN-0": 0.16},
    "Er5Tl3": {"OMAT-0-medium": 0.18, "MATPES-PBE-0": 0.17, "MATPES-r2SCAN-0": 0.13},
}
# runs with imaginary modes on the band path, and runs whose trajectory check
# was not "stable"
IMAGINARY = {
    ("AlAsPt5", "MATPES-PBE-0"): (100,),
    ("Bi4S3N2", "OMAT-0-medium"): (100, 300, 500),
    ("Bi4S3N2", "MATPES-PBE-0"): (300, 500, 700, 900),
    ("Bi4S3N2", "MATPES-r2SCAN-0"): (100, 300, 500, 700, 900),
}
NOT_STABLE = {
    ("Ca3Ir4Sn13", "OMAT-0-medium", 100): "shifted_or_diffusing",
    ("Ca3Ir4Sn13", "MATPES-r2SCAN-0", 100): "shifted_or_diffusing",
    ("Er5Tl3", "MATPES-r2SCAN-0", 100): "shifted_or_diffusing",
    ("Zn3P2", "OMAT-0-medium", 900): "melted",
    ("Zn3P2", "MATPES-PBE-0", 900): "melted",
}

fig, axes = plt.subplots(1, 5, figsize=(16, 3.4))
for ax, (name, by_model) in zip(axes, MAX_FREQUENCY.items(), strict=True):
    for model, values in by_model.items():
        ax.plot(TEMPERATURES, values, "o-", label=model)
    ax.axhline(AIMD_MAX_300K[name], color="black", ls="--", lw=0.8)
    ax.set_title(name)
    ax.set_xlabel("Temperature (K)")
axes[0].set_ylabel("Highest frequency (THz)")
fig.tight_layout()
fig.legend(
    *axes[0].get_legend_handles_labels(),
    loc="upper center",
    ncol=3,
    bbox_to_anchor=(0.5, 0),
)
plt.show()

pd.DataFrame(MAE_300K).T.assign(**{"AIMD highest frequency": pd.Series(AIMD_MAX_300K)})

![Highest frequency against temperature](finite_temperature_phonons_figures/s1_max_frequency.png)

| Material | OMAT-0-medium | MATPES-PBE-0 | MATPES-r2SCAN-0 | AIMD highest frequency (THz) |
|---|---|---|---|---|
| Ca3Ir4Sn13 | 0.07 | 0.17 | 0.10 | 6.03 |
| AlAsPt5 | 0.11 | 0.19 | 0.17 | 7.79 |
| Zn3P2 | 0.21 | 0.10 | 0.44 | 10.09 |
| Bi4S3N2 | 0.15 | 0.46 | 0.16 | 17.03 |
| Er5Tl3 | 0.18 | 0.17 | 0.13 | 3.97 |

The table gives the mean absolute difference from AIMD at 300 K over the band
path, in THz.

- No potential is best for all materials. MACE-OMAT-0-medium is closest for
  three of the five.
- Most frequencies soften with temperature. The highest frequency of Bi4S3N2
  drops by 2.2 THz from 100 to 900 K. AlAsPt5 barely changes.
- The imaginary modes of 0 K are gone for Ca3Ir4Sn13, Zn3P2 and Er5Tl3. Bi4S3N2
  keeps some in 12 of 15 runs, and so does its AIMD reference.
- Zn3P2 melts at 900 K with two of the potentials. At 100 K three runs are
  reported as shifted. Their atoms likely sit in a slightly distorted
  structure.

## 2. Thermal expansion with an NPT MD

Each force field has its own 0 K cell, and the crystal expands as it heats up.
With `run_npt=True`, the workflow relaxes the structure and runs an NPT MD at
the temperature, 8 ps by default. It averages the cell after the first 2 ps,
keeping the symmetry, and relaxes the atoms in that cell. The NVT MD and the fit
then use this cell. We run Ca3Ir4Sn13 at 900 K.

In [ ]:
maker = ForceFieldFiniteTemperaturePhononMaker.from_force_field_name(
    "MACE-MP-0",
    calculator_kwargs=calc_kwargs("OMAT-0-medium"),
    run_npt=True,
    temperature=900,
)
flow = maker.make(structure, supercell_matrix=[[2, 0, 0], [0, 2, 0], [0, 0, 2]])
responses = run_locally(
    flow,
    store=JobStore(MemoryStore(), additional_stores={"data": MemoryStore()}),
    create_folders=True,
    ensure_success=True,
)
doc_npt = responses[flow.output.uuid][1].output
{
    "lattice at 0 K in A": doc_npt.npt_input_structure.lattice.abc,
    "lattice at 900 K in A": doc_npt.structure.lattice.abc,
    "volume ratio": doc_npt.structure.volume / doc_npt.npt_input_structure.volume,
    "NPT trajectory check": doc_npt.npt_trajectory_health.verdict,
    "highest frequency in THz": float(np.max(doc_npt.phonon_bandstructure.bands)),
    "imaginary modes on the band path": doc_npt.has_imaginary_modes,
    "trajectory check": doc_npt.trajectory_health.verdict,
}

Our run took about 20 minutes. The 0 K cell of MACE-OMAT-0-medium is 3.8% larger
than the PBEsol cell, and the 900 K cell 4.3% larger again. Both trajectories
were stable. The highest frequency is 4.91 THz, against 5.78 THz in the PBEsol
cell.

## All runs

The loop runs the 75 settings of Section 1 again, in the force field's 0 K cell
and with the NPT MD. It is off by default.

In [ ]:
RUN_NPT = False
if RUN_NPT:
    for name in ("Ca3Ir4Sn13", "AlAsPt5", "Zn3P2", "Bi4S3N2", "Er5Tl3"):
        mp_id, supercell = MATERIALS[name]
        unit_cell = get_pheasy_structure(mp_id)
        for model in MODEL_FILES:
            for temperature in TEMPERATURES:
                for label, run_npt in (("relaxed", False), ("npt", True)):
                    docs[name, model, temperature, label] = run_ft(
                        unit_cell,
                        supercell,
                        model,
                        f"runs/{name}_{model}_{temperature}K_{label}",
                        temperature,
                        relax=True,
                        run_npt=run_npt,
                    )

The next cell plots the bands and the DOS in the NPT cell at 100, 500 and
900 K, as in Section 1.

In [ ]:
if RUN_NPT:
    for name in ("Ca3Ir4Sn13", "AlAsPt5", "Zn3P2", "Bi4S3N2", "Er5Tl3"):
        plot_temperatures(name, label="npt")

The figures below show our runs.

![Band structures and DOS of Ca3Ir4Sn13 at 0 K with PBEsol and in the NPT cell at 100, 500 and 900 K](finite_temperature_phonons_figures/s2_bands_Ca3Ir4Sn13.png)

![Band structures and DOS of AlAsPt5 at 0 K with PBEsol and in the NPT cell at 100, 500 and 900 K](finite_temperature_phonons_figures/s2_bands_AlAsPt5.png)

![Band structures and DOS of Zn3P2 at 0 K with PBEsol and in the NPT cell at 100, 500 and 900 K](finite_temperature_phonons_figures/s2_bands_Zn3P2.png)

![Band structures and DOS of Bi4S3N2 at 0 K with PBEsol and in the NPT cell at 100, 500 and 900 K](finite_temperature_phonons_figures/s2_bands_Bi4S3N2.png)

![Band structures and DOS of Er5Tl3 at 0 K with PBEsol and in the NPT cell at 100, 500 and 900 K](finite_temperature_phonons_figures/s2_bands_Er5Tl3.png)

- From 100 to 900 K the highest frequency drops by 0.3 to 3.2 THz in the NPT
  cell, against at most 2.3 THz in the PBEsol cell of Section 1.
- AlAsPt5 barely changes in the PBEsol cell, but softens by 0.3 to 0.6 THz in
  the NPT cell.

## Our results

The plots give the volume relative to the PBEsol cell, and the highest frequency
in three cells: the PBEsol cell, the force field's 0 K cell and the NPT cell.

In [ ]:
# volume relative to the PBEsol cell: the force field's relaxed 0 K cell, then
# the NPT cell at 100, 300, 500, 700 and 900 K
VOLUME = {
    "Ca3Ir4Sn13": {
        "OMAT-0-medium": (1.0382, 1.0420, 1.0517, 1.0612, 1.0716, 1.0828),
        "MATPES-PBE-0": (1.0432, 1.0475, 1.0568, 1.0665, 1.0774, 1.0890),
        "MATPES-r2SCAN-0": (1.0192, 1.0227, 1.0311, 1.0398, 1.0491, 1.0591),
    },
    "AlAsPt5": {
        "OMAT-0-medium": (1.0341, 1.0382, 1.0463, 1.0551, 1.0642, 1.0744),
        "MATPES-PBE-0": (1.0175, 1.0229, 1.0313, 1.0400, 1.0501, 1.0501),
        "MATPES-r2SCAN-0": (1.0021, 1.0061, 1.0133, 1.0210, 1.0302, 1.0392),
    },
    "Zn3P2": {
        "OMAT-0-medium": (1.0533, 1.0585, 1.0681, 1.0780, 1.0884, 1.1012),
        "MATPES-PBE-0": (1.0510, 1.0551, 1.0636, 1.0723, 1.0813, 1.0934),
        "MATPES-r2SCAN-0": (1.0087, 1.0125, 1.0211, 1.0301, 1.0387, 1.0491),
    },
    "Bi4S3N2": {
        "OMAT-0-medium": (1.1075, 1.1153, 1.1269, 1.1424, 1.1515, 1.1741),
        "MATPES-PBE-0": (1.0457, 1.0594, 1.0764, 1.0967, 1.1515, 1.1625),
        "MATPES-r2SCAN-0": (1.0292, 1.0381, 1.0502, 1.0656, 1.0807, 1.0861),
    },
    "Er5Tl3": {
        "OMAT-0-medium": (1.0291, 1.0356, 1.0465, 1.0573, 1.0684, 1.0801),
        "MATPES-PBE-0": (1.1224, 1.1283, 1.1431, 1.1589, 1.1744, 1.1939),
        "MATPES-r2SCAN-0": (1.0646, 1.0697, 1.0806, 1.0925, 1.1034, 1.1164),
    },
}
# highest frequency on the band path in THz, at 100, 300, 500, 700 and 900 K,
# in the force field's relaxed 0 K cell
MAX_RELAXED = {
    "Ca3Ir4Sn13": {
        "OMAT-0-medium": (5.58, 5.58, 5.51, 5.47, 5.40),
        "MATPES-PBE-0": (5.37, 5.31, 5.26, 5.19, 5.17),
        "MATPES-r2SCAN-0": (5.78, 5.76, 5.69, 5.59, 5.54),
    },
    "AlAsPt5": {
        "OMAT-0-medium": (7.19, 7.07, 7.09, 7.20, 7.17),
        "MATPES-PBE-0": (7.23, 7.24, 7.11, 7.09, 7.30),
        "MATPES-r2SCAN-0": (7.52, 7.43, 7.42, 7.49, 7.45),
    },
    "Zn3P2": {
        "OMAT-0-medium": (10.25, 9.82, 9.67, 9.39, 8.91),
        "MATPES-PBE-0": (9.99, 9.73, 9.35, 9.06, 8.70),
        "MATPES-r2SCAN-0": (11.19, 10.92, 10.46, 10.40, 9.84),
    },
    "Bi4S3N2": {
        "OMAT-0-medium": (16.74, 15.80, 15.47, 14.80, 14.41),
        "MATPES-PBE-0": (15.40, 15.02, 14.32, 14.55, 14.42),
        "MATPES-r2SCAN-0": (17.27, 16.95, 16.22, 15.74, 15.42),
    },
    "Er5Tl3": {
        "OMAT-0-medium": (4.10, 4.07, 4.06, 4.00, 4.01),
        "MATPES-PBE-0": (3.70, 3.62, 3.56, 3.57, 3.40),
        "MATPES-r2SCAN-0": (3.83, 3.83, 3.77, 3.78, 3.72),
    },
}
# the same in the NPT cell
MAX_NPT = {
    "Ca3Ir4Sn13": {
        "OMAT-0-medium": (5.56, 5.39, 5.26, 5.11, 4.91),
        "MATPES-PBE-0": (5.33, 5.19, 5.03, 4.89, 4.71),
        "MATPES-r2SCAN-0": (5.72, 5.59, 5.44, 5.26, 5.06),
    },
    "AlAsPt5": {
        "OMAT-0-medium": (7.18, 6.90, 6.90, 6.51, 6.88),
        "MATPES-PBE-0": (7.15, 6.94, 6.90, 6.63, 6.64),
        "MATPES-r2SCAN-0": (7.39, 7.27, 7.26, 7.11, 6.77),
    },
    "Zn3P2": {
        "OMAT-0-medium": (10.23, 9.82, 9.31, 8.91, 8.42),
        "MATPES-PBE-0": (9.97, 9.68, 9.19, 8.86, 8.19),
        "MATPES-r2SCAN-0": (11.16, 10.72, 10.39, 9.87, 9.66),
    },
    "Bi4S3N2": {
        "OMAT-0-medium": (16.39, 15.58, 15.15, 13.75, 13.63),
        "MATPES-PBE-0": (15.50, 15.04, 14.74, 13.39, 12.34),
        "MATPES-r2SCAN-0": (17.38, 16.63, 16.28, 15.33, 14.67),
    },
    "Er5Tl3": {
        "OMAT-0-medium": (4.04, 3.99, 3.79, 3.77, 3.66),
        "MATPES-PBE-0": (3.67, 3.56, 3.46, 3.32, 3.25),
        "MATPES-r2SCAN-0": (3.78, 3.72, 3.64, 3.56, 3.44),
    },
}

fig, axes = plt.subplots(1, 5, figsize=(16, 3.4))
for ax, (name, by_model) in zip(axes, VOLUME.items(), strict=True):
    for model, values in by_model.items():
        ax.plot((0, *TEMPERATURES), values, "o-", label=model)
    ax.axhline(1, color="black", ls="--", lw=0.8)
    ax.set_title(name)
    ax.set_xlabel("Temperature (K)")
axes[0].set_ylabel("Volume / PBEsol volume")
fig.tight_layout()
fig.legend(
    *axes[0].get_legend_handles_labels(),
    loc="upper center",
    ncol=3,
    bbox_to_anchor=(0.5, 0),
)
plt.show()

fig, axes = plt.subplots(3, 5, figsize=(16, 8.5), sharex=True)
for col, name in enumerate(VOLUME):
    for row, model in enumerate(MODEL_FILES):
        ax = axes[row, col]
        for label, values, style in (
            ("PBEsol cell", MAX_FREQUENCY, "o-"),
            ("relaxed 0 K cell", MAX_RELAXED, "s--"),
            ("NPT cell", MAX_NPT, "^:"),
        ):
            ax.plot(TEMPERATURES, values[name][model], style, label=label)
        if row == 0:
            ax.set_title(name)
        if col == 0:
            ax.set_ylabel(f"{model}\nHighest frequency (THz)")
        if row == 2:
            ax.set_xlabel("Temperature (K)")
fig.tight_layout()
fig.legend(
    *axes[0, 0].get_legend_handles_labels(),
    loc="upper center",
    ncol=3,
    bbox_to_anchor=(0.5, 0),
)
plt.show()

![Volume against temperature](finite_temperature_phonons_figures/s2_volume.png)

![Highest frequency in three cells](finite_temperature_phonons_figures/s2_max_frequency.png)

- The 0 K cells of the force fields are up to 12% larger than the PBEsol cell.
  MACE-MATPES-r2SCAN-0 is closest to PBEsol for four of the five materials.
- From 0 to 900 K the volume grows by 3 to 6.5%, and by 11% for Bi4S3N2 with
  MACE-MATPES-PBE-0.
- The larger 0 K cell alone lowers the frequencies, by up to 0.5 THz at 300 K.
- Thermal expansion lowers them by 0.15 to 0.8 THz at 900 K, and by 2.1 THz for
  Bi4S3N2 with MACE-MATPES-PBE-0.
- Zn3P2 and Bi4S3N2 melt at 900 K in the NPT runs. AlAsPt5 with
  MACE-MATPES-PBE-0 is reported as shifted at 900 K, and its cell does not grow
  from 700 to 900 K.

To compare with DFT in a given cell, run in that cell as in Section 1. To
predict the phonons at a temperature with the force field alone, use
`run_npt=True`.

## 3. MD length and number of snapshots

We run KNaICl (mp-1002081, 72 atoms in the supercell) at 300 K with 4, 8 and 16 ps
of MD and 25, 50 and 100 snapshots, and compare with AIMD. KNaICl is polar, so
the comparison is without the non-analytical correction. The grid is off by
default.

In [ ]:
RUN_GRID = False
if RUN_GRID:
    mp_id, supercell = MATERIALS["KNaICl"]
    knaicl = get_pheasy_structure(mp_id)
    for model in MODEL_FILES:
        for md_time in (4, 8, 16):
            for n_snapshots in (25, 50, 100):
                docs["KNaICl", model, md_time, n_snapshots] = run_ft(
                    knaicl,
                    supercell,
                    model,
                    f"runs/KNaICl_{model}_{md_time}ps_{n_snapshots}",
                    300,
                    md_time,
                    n_snapshots,
                )

In [ ]:
MD_TIMES = (4, 8, 16)
N_SNAPSHOTS = (25, 50, 100)
# by potential, rows 4, 8 and 16 ps, columns 25, 50 and 100 snapshots
MAE_GRID = {  # mean absolute difference from AIMD at 300 K in THz
    "OMAT-0-medium": (
        (0.179, 0.181, 0.147),
        (0.167, 0.170, 0.148),
        (0.162, 0.171, 0.158),
    ),
    "MATPES-PBE-0": (
        (0.158, 0.137, 0.133),
        (0.129, 0.108, 0.120),
        (0.142, 0.089, 0.105),
    ),
    "MATPES-r2SCAN-0": (
        (0.188, 0.161, 0.162),
        (0.191, 0.184, 0.136),
        (0.152, 0.154, 0.169),
    ),
}
MAX_GRID = {  # highest frequency on the band path in THz
    "OMAT-0-medium": ((5.48, 5.52, 5.66), (5.35, 5.56, 5.63), (5.57, 5.62, 5.61)),
    "MATPES-PBE-0": ((5.20, 5.18, 5.13), (5.07, 5.22, 5.33), (5.05, 5.31, 5.24)),
    "MATPES-r2SCAN-0": ((5.64, 5.51, 5.57), (5.32, 5.58, 5.42), (5.42, 5.53, 5.53)),
}
IMAGINARY_GRID = {  # True if the band path has imaginary modes
    "OMAT-0-medium": ((False, True, True), (True, True, False), (True, True, False)),
    "MATPES-PBE-0": ((True, True, True), (False, True, False), (False, False, True)),
    "MATPES-r2SCAN-0": ((True, True, True), (True, True, False), (True, True, True)),
}

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, model in zip(axes, MODEL_FILES, strict=True):
    grid = np.array(MAE_GRID[model])
    image = ax.imshow(grid, cmap="viridis", vmin=0.08, vmax=0.20)
    for i in range(3):
        for j in range(3):
            mark = " *" if IMAGINARY_GRID[model][i][j] else ""
            ax.text(
                j, i, f"{grid[i, j]:.3f}{mark}", ha="center", va="center", color="w"
            )
    ax.set_xticks(range(3), N_SNAPSHOTS)
    ax.set_yticks(range(3), [f"{t} ps" for t in MD_TIMES])
    ax.set_xlabel("Number of snapshots")
    ax.set_title(model)
fig.colorbar(image, ax=axes, label="Mean absolute difference from AIMD (THz)")
plt.show()

pd.concat(
    {
        model: pd.DataFrame(
            MAX_GRID[model], index=[f"{t} ps" for t in MD_TIMES], columns=N_SNAPSHOTS
        )
        for model in MODEL_FILES
    },
    axis=1,
)

![Difference from AIMD against MD length and snapshots](finite_temperature_phonons_figures/s3_knaicl_grid.png)

| Potential | MD length | 25 snapshots | 50 snapshots | 100 snapshots |
|---|---|---|---|---|
| OMAT-0-medium | 4 ps | 5.48 | 5.52 | 5.66 |
| OMAT-0-medium | 8 ps | 5.35 | 5.56 | 5.63 |
| OMAT-0-medium | 16 ps | 5.57 | 5.62 | 5.61 |
| MATPES-PBE-0 | 4 ps | 5.20 | 5.18 | 5.13 |
| MATPES-PBE-0 | 8 ps | 5.07 | 5.22 | 5.33 |
| MATPES-PBE-0 | 16 ps | 5.05 | 5.31 | 5.24 |
| MATPES-r2SCAN-0 | 4 ps | 5.64 | 5.51 | 5.57 |
| MATPES-r2SCAN-0 | 8 ps | 5.32 | 5.58 | 5.42 |
| MATPES-r2SCAN-0 | 16 ps | 5.42 | 5.53 | 5.53 |

A star marks a run with imaginary modes on the band path. The table gives the
highest frequency in THz.

- The difference from AIMD is 0.09 to 0.19 THz, with no trend. It is the noise
  of a single trajectory.
- 19 of the 27 runs have imaginary modes, mostly at q-points between those of
  the supercell, where the frequencies are interpolated. AIMD has none. An
  earlier round with a Nose-Hoover thermostat gave 12 of 27.
- More snapshots cost little with a force field, but they did not remove these
  modes. The defaults stay at 8 ps and 50 snapshots, since with VASP each
  snapshot is a DFT calculation.

## 4. Li3PS4, a crystal that is unstable at 0 K

In experiment Li3PS4 is γ up to 523 to 573 K, β up to 723 to 748 K and α above
(Homma et al., Solid State Ionics 182, 53 (2011); Kaup et al., J. Mater. Chem. A
8, 12446 (2020)). mp-985583 is one ordering of the partly occupied Li sites of
β, and it has imaginary modes at 0 K. We run it from 100 to 1000 K with each
potential, in the relaxed 0 K cell (NVT) and in the NPT cell. The supercell has
128 atoms.

In [ ]:
li3ps4 = get_pheasy_structure("mp-985583")
LI3PS4_SUPERCELL = (2, 2, 1)  # 128 atoms
LI3PS4_TEMPERATURES = tuple(range(100, 1001, 100))
li3ps4.composition, li3ps4.get_space_group_info()

## Phonons at 0 K

We use the phonopy workflow here, since the pheasy workflow refits the force
constants to remove imaginary modes. The relaxation keeps the symmetry.

In [ ]:
from atomate2.common.schemas.phonons import PhononBSDOSDoc
from atomate2.forcefields.flows.phonons import PhononMaker


def li3ps4_harmonic(model: str) -> PhononBSDOSDoc:
    """Run the phonopy workflow at 0 K, in the cell relaxed by the potential."""
    maker = PhononMaker.from_force_field_name(
        "MACE-MP-0", calculator_kwargs=calc_kwargs(model)
    )
    maker.bulk_relax_maker.fix_symmetry = True
    flow = maker.make(li3ps4, supercell_matrix=np.diag(LI3PS4_SUPERCELL).tolist())
    root_dir = f"runs/Li3PS4_{model}_0K"
    os.makedirs(root_dir, exist_ok=True)
    responses = run_locally(
        flow,
        store=JobStore(MemoryStore(), additional_stores={"data": MemoryStore()}),
        create_folders=True,
        root_dir=root_dir,
        ensure_success=True,
    )
    return responses[flow.output.uuid][1].output


doc_0k = li3ps4_harmonic("OMAT-0-medium")
float(np.min(doc_0k.phonon_bandstructure.bands))

Our run took 2 minutes. The lowest frequency is -2.59 THz. With
MACE-MATPES-PBE-0 it is -1.97 THz and with MACE-MATPES-r2SCAN-0 -3.48 THz.

## One temperature

We run 600 K with MACE-OMAT-0-medium and the NPT MD.

In [ ]:
doc_li3ps4 = run_ft(
    li3ps4,
    LI3PS4_SUPERCELL,
    "OMAT-0-medium",
    "runs/Li3PS4_OMAT-0-medium_600K_npt",
    600,
    relax=True,
    run_npt=True,
)
{
    "volume at 0 K in A^3": doc_li3ps4.npt_input_structure.volume,
    "volume at 600 K in A^3": doc_li3ps4.structure.volume,
    "lowest frequency in THz": float(np.min(doc_li3ps4.phonon_bandstructure.bands)),
    "trajectory check": doc_li3ps4.trajectory_health.verdict,
    "Lindemann ratio": doc_li3ps4.trajectory_health.lindemann_ratio,
}

Our run took 31 minutes. The volume grows from 664.8 to 698.5 Å³. The lowest
frequency is -0.57 THz, and the check reports "melted" with a Lindemann ratio of
0.24. Section 5 shows that only the Li atoms move.

## All runs

The loop runs the 60 settings and the three 0 K runs. It is off by default. Each
run took 15 to 62 minutes.

In [ ]:
RUN_LI3PS4 = False
li3ps4_docs = {}
if RUN_LI3PS4:
    for model in MODEL_FILES:
        li3ps4_docs[model, 0, "harmonic"] = li3ps4_harmonic(model)
        for temperature in LI3PS4_TEMPERATURES:
            for label, run_npt in (("nvt", False), ("npt", True)):
                li3ps4_docs[model, temperature, label] = run_ft(
                    li3ps4,
                    LI3PS4_SUPERCELL,
                    model,
                    f"runs/Li3PS4_{model}_{temperature}K_{label}",
                    temperature,
                    relax=True,
                    run_npt=run_npt,
                )

The next cell plots the bands and the DOS at 0, 300 and 700 K. Black is PBEsol at
0 K from the Materials Project's Harmonic Phonon Database, gray the potential at
0 K. Every run keeps the Pnma symmetry, so all runs share the band path of the
potential's 0 K run.

In [ ]:
def plot_li3ps4_bands(model: str) -> None:
    """Plot the bands and DOS of one potential at 0, 300 and 700 K, in both cells."""
    bs_pbesol, dos_pbesol = get_pheasy_phonons("mp-985583")
    harmonic = li3ps4_docs[model, 0, "harmonic"]
    reference = harmonic.phonon_bandstructure
    _, axes = plt.subplots(
        1, 4, figsize=(16, 4), sharey=True, gridspec_kw={"width_ratios": [3, 1, 3, 1]}
    )
    for idx, label in enumerate(("nvt", "npt")):
        plot_bands(
            axes[2 * idx],
            bs_pbesol,
            label="0 K, PBEsol",
            distance=reference.distance,
            color="black",
            lw=0.5,
        )
        plot_dos(axes[2 * idx + 1], dos_pbesol, label="0 K, PBEsol", color="black")
        plot_bands(axes[2 * idx], reference, label="0 K, MACE", color="gray", lw=0.5)
        plot_dos(
            axes[2 * idx + 1], harmonic.phonon_dos, label="0 K, MACE", color="gray"
        )
        for temperature, color in ((300, "C0"), (700, "C3")):
            doc = li3ps4_docs[model, temperature, label]
            plot_bands(
                axes[2 * idx],
                doc.phonon_bandstructure,
                label=f"{temperature} K",
                distance=reference.distance,
                color=color,
                lw=0.5,
            )
            plot_dos(
                axes[2 * idx + 1], doc.phonon_dos, label=f"{temperature} K", color=color
            )
        axes[2 * idx].set_title(f"{model}, {label.upper()}")
    axes[0].set_ylabel("Frequency (THz)")
    axes[-1].legend(fontsize=8)
    plt.show()


if RUN_LI3PS4:
    for model in MODEL_FILES:
        plot_li3ps4_bands(model)

The figures below show our runs.

![Band structures and DOS of Li3PS4 at 0 K with PBEsol and with OMAT-0-medium at 0, 300 and 700 K](finite_temperature_phonons_figures/s4_bands_OMAT-0-medium.png)

![Band structures and DOS of Li3PS4 at 0 K with PBEsol and with MATPES-PBE-0 at 0, 300 and 700 K](finite_temperature_phonons_figures/s4_bands_MATPES-PBE-0.png)

![Band structures and DOS of Li3PS4 at 0 K with PBEsol and with MATPES-r2SCAN-0 at 0, 300 and 700 K](finite_temperature_phonons_figures/s4_bands_MATPES-r2SCAN-0.png)

## Our results

The lowest frequency, the Lindemann ratio and the NPT volume of each run.

In [ ]:
# lowest frequency on the band path in THz and Lindemann ratio of the NVT MD,
# at 100 to 1000 K, in the relaxed 0 K cell (nvt) and in the NPT cell (npt)
LOWEST_FREQUENCY = {
    "nvt": {
        "OMAT-0-medium": (-0.8, -0.4, -0.9, -0.8, -0.4, -1.6, -1.1, -1.1, -1.5, -1.5),
        "MATPES-PBE-0": (-0.2, -0.9, -1.0, -2.2, -1.4, -1.3, -1.7, -1.6, -1.5, -1.6),
        "MATPES-r2SCAN-0": (-0.7, -0.3, -0.6, -0.3, -0.7, -1.1, -1.6, -1.5, -1.5, -1.4),
    },
    "npt": {
        "OMAT-0-medium": (-0.7, -0.4, -0.7, -0.7, -0.5, -0.6, -1.4, -1.5, -1.3, -1.4),
        "MATPES-PBE-0": (-0.4, -0.5, -1.4, -0.5, -1.3, -1.4, -1.3, -1.7, -1.6, -1.6),
        "MATPES-r2SCAN-0": (-0.5, -0.6, -1.4, 0.0, -0.7, -1.0, -1.1, -1.3, -1.8, -1.6),
    },
}
LINDEMANN = {
    "nvt": {
        "OMAT-0-medium": (0.06, 0.1, 0.12, 0.17, 0.18, 0.26, 0.28, 0.33, 0.51, 0.54),
        "MATPES-PBE-0": (0.08, 0.13, 0.16, 0.2, 0.24, 0.36, 0.42, 0.41, 0.55, 0.84),
        "MATPES-r2SCAN-0": (0.06, 0.1, 0.13, 0.17, 0.21, 0.26, 0.28, 0.36, 0.58, 0.45),
    },
    "npt": {
        "OMAT-0-medium": (0.08, 0.11, 0.15, 0.17, 0.19, 0.24, 0.39, 0.36, 0.58, 0.77),
        "MATPES-PBE-0": (0.08, 0.13, 0.18, 0.19, 0.23, 0.32, 0.33, 0.41, 0.61, 0.74),
        "MATPES-r2SCAN-0": (0.07, 0.1, 0.13, 0.18, 0.21, 0.24, 0.27, 0.4, 0.52, 0.61),
    },
}
# volume of the unit cell in A^3, relaxed at 0 K and from the NPT MD
VOLUME_0K = {"OMAT-0-medium": 664.7, "MATPES-PBE-0": 651.6, "MATPES-r2SCAN-0": 649.6}
NPT_VOLUME = {
    "OMAT-0-medium": (670, 675, 680, 687, 693, 698, 702, 713, 776, 767),
    "MATPES-PBE-0": (656, 658, 661, 660, 663, 662, 667, 672, 678, 692),
    "MATPES-r2SCAN-0": (656, 661, 666, 669, 673, 673, 675, 683, 691, 699),
}

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for idx, model in enumerate(MODEL_FILES):
    for label, style in (("nvt", "-o"), ("npt", "--s")):
        kwargs = {"color": f"C{idx}", "ms": 3, "label": f"{model}, {label}"}
        axes[0].plot(
            LI3PS4_TEMPERATURES, LOWEST_FREQUENCY[label][model], style, **kwargs
        )
        axes[1].plot(LI3PS4_TEMPERATURES, LINDEMANN[label][model], style, **kwargs)
    ratio = np.array(NPT_VOLUME[model]) / VOLUME_0K[model]
    axes[2].plot(LI3PS4_TEMPERATURES, ratio, "--s", color=f"C{idx}", ms=3)
axes[1].axhline(0.15, color="gray", ls=":")
axes[0].set_ylabel("Lowest frequency (THz)")
axes[1].set_ylabel("Lindemann ratio")
axes[2].set_ylabel("V(NPT) / V(0 K)")
for ax in axes:
    ax.set_xlabel("Temperature (K)")
axes[0].legend(fontsize=7)
fig.tight_layout()
plt.show()

![Lowest frequency, Lindemann ratio and volume of Li3PS4](finite_temperature_phonons_figures/s4_runs.png)

- The imaginary modes shrink with temperature but do not go away. The one
  exception is MACE-MATPES-r2SCAN-0 at 400 K in the NPT cell.
- The Lindemann ratio passes 0.15 at 300 to 400 K, and the check reports
  melting. Li3PS4 does not melt there in experiment.
- The NPT cell grows by 3 to 7% up to 800 K.
- MACE-MATPES-PBE-0 gives the softest bands, with 15.4 THz at 0 K against 17.8
  and 17.9 THz.

## 5. Li3PS4 with hopping Li

At 0 K the imaginary modes come from the structure, since this ordering is not
at an energy minimum. Above about 500 K the Li atoms also start to hop, while
the PS4 framework stays solid. A fit around fixed sites reads this as very soft
springs. The MD itself needs no fixed sites. We get the diffusion, the
vibrational DOS and the free energy from the saved trajectories,
`md_trajectory.traj`, starting with the 600 K run above.

In [ ]:
from pathlib import Path

from ase import Atoms
from ase.io import Trajectory

SKIP = 1000  # frames left out at the start, as in the fit
TIME_STEP = 1e-3  # ps per frame


def nvt_trajectory(folder: str) -> list[Atoms]:
    """Get the frames of the NVT MD in a run folder, after the first SKIP."""
    for path in Path(folder).rglob("md_trajectory.traj"):
        traj = Trajectory(str(path))
        if np.allclose(traj[0].cell.array, traj[-1].cell.array):
            return [traj[i] for i in range(SKIP, len(traj))]
    raise FileNotFoundError(f"no NVT trajectory in {folder}")


def unwrapped_positions(frames: list[Atoms]) -> np.ndarray:
    """Get the Cartesian positions of every frame, without jumps across the cell."""
    cell = frames[0].cell.array
    frac = np.array([atoms.get_scaled_positions(wrap=False) for atoms in frames])
    steps = np.diff(frac, axis=0)
    steps -= np.round(steps)
    frac = np.concatenate([frac[:1], frac[:1] + np.cumsum(steps, axis=0)])
    return frac @ cell


frames_600k = nvt_trajectory("runs/Li3PS4_OMAT-0-medium_600K_npt")

## Diffusion

$D$ is the slope of the mean-square displacement divided by 6, fitted from 1 to
3 ps.

In [ ]:
def diffusion(
    frames: list[Atoms], max_lag: int = 3000, fit: tuple[int, int] = (1000, 3000)
) -> dict[str, tuple[np.ndarray, np.ndarray, float]]:
    """Get the MSD (A^2) against time (ps) and D (cm^2/s) of each element.

    The motion of the center of mass is removed. fit gives the range of lags in
    frames where the slope of the MSD is fitted.
    """
    masses = frames[0].get_masses()
    symbols = np.array(frames[0].get_chemical_symbols())
    pos = unwrapped_positions(frames)
    pos -= ((pos * masses[None, :, None]).sum(1) / masses.sum())[:, None]
    lags = np.arange(0, max_lag + 1, 10)
    origins = np.arange(0, len(pos) - max_lag, 100)
    msd = np.array(
        [((pos[origins + lag] - pos[origins]) ** 2).sum(2).mean(0) for lag in lags]
    )
    keep = (lags >= fit[0]) & (lags <= fit[1])
    out = {}
    for element in ("Li", "P", "S"):
        curve = msd[:, symbols == element].mean(1)
        slope = np.polyfit(lags[keep] * TIME_STEP, curve[keep], 1)[0]
        out[element] = (lags * TIME_STEP, curve, slope / 6 * 1e-4)
    return out


{element: f"{d:.1e}" for element, (_, _, d) in diffusion(frames_600k).items()}

In [ ]:
# diffusion coefficient in cm^2/s of each element in the NVT MD in the relaxed
# 0 K cell, at 300 to 1000 K
D_TEMPERATURES = (300, 400, 500, 600, 700, 800, 900, 1000)
DIFFUSION = {
    "OMAT-0-medium": {
        "Li": (-4.4e-07, 2e-06, -4.9e-08, 7.6e-06, 8e-06, 5e-06, 3.1e-05, 4.7e-05),
        "P": (-1.3e-08, -3.3e-08, 1.2e-08, 1.4e-07, 1.3e-07, 6.9e-08, 9.6e-07, 2.6e-07),
        "S": (-3.3e-08, -1.9e-08, 2.1e-08, 2.4e-07, 2.1e-07, 4.7e-07, 1.9e-06, 1.3e-06),
    },
    "MATPES-PBE-0": {
        "Li": (1.9e-06, 3e-06, 6.3e-06, 1.5e-05, 2.6e-05, 2.9e-05, 4.6e-05, 6.4e-05),
        "P": (4.6e-08, 2e-08, -1.9e-08, 1.4e-07, -2.6e-07, 2.9e-07, 1.2e-07, 3.5e-07),
        "S": (1.2e-07, 1.8e-08, 3.9e-08, 2.7e-07, -3.7e-07, 5.4e-07, -2.7e-07, 8.1e-07),
    },
    "MATPES-r2SCAN-0": {
        "Li": (1.6e-06, 7.9e-07, 2.6e-06, 4.8e-06, 6.8e-06, 2.6e-05, 3.8e-05, 3.4e-05),
        "P": (
            3.2e-09,
            -4.1e-09,
            -6.6e-08,
            8.4e-08,
            1.5e-07,
            1.4e-07,
            4.2e-07,
            -2.2e-07,
        ),
        "S": (1e-07, 9.1e-09, 7e-08, 1.8e-07, 7e-08, 3e-07, 9.3e-07, -3.7e-09),
    },
}

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
inverse_t = 1000 / np.array(D_TEMPERATURES)
for ax, model in zip(axes, MODEL_FILES, strict=True):
    ax.axvspan(1000 / 748, 1000 / 523, color="0.9")
    for element, color in (("Li", "C0"), ("P", "C2"), ("S", "C3")):
        d = np.array(DIFFUSION[model][element])
        keep = d > 1e-8
        ax.semilogy(inverse_t[keep], d[keep], "-o", color=color, ms=3, label=element)
    ax.axhline(3e-7, color="gray", ls=":")
    ax.set_title(model)
    ax.set_xlabel("1000 / T (1/K)")
axes[0].set_ylabel("D (cm$^2$/s)")
axes[0].legend()
fig.tight_layout()
plt.show()

![Diffusion coefficients of Li3PS4](finite_temperature_phonons_figures/s5_diffusion.png)

D of Li in 10⁻⁶ cm²/s, in the relaxed 0 K cell:

| Potential | 300 K | 400 K | 500 K | 600 K | 700 K | 800 K | 900 K | 1000 K |
|---|---|---|---|---|---|---|---|---|
| OMAT-0-medium | -0.4 | 2.0 | 0.0 | 7.6 | 8.0 | 5.0 | 31.0 | 47.0 |
| MATPES-PBE-0 | 1.9 | 3.0 | 6.3 | 15.0 | 26.0 | 29.0 | 46.0 | 64.0 |
| MATPES-r2SCAN-0 | 1.6 | 0.8 | 2.6 | 4.8 | 6.8 | 26.0 | 38.0 | 34.0 |

A $D$ below 3×10⁻⁷ cm²/s is noise in these 7 ps runs.

- Li diffuses from about 600 K, and from about 500 K with MACE-MATPES-PBE-0.
- P and S start to diffuse at 900 to 1000 K.
- $D$ depends on the Langevin friction. For transport, run an NVE MD.

## Vibrational DOS

The DOS is the mass-weighted power spectrum of the velocities,

$$g(\nu) \propto \sum_i m_i \left| \int v_i(t)\, e^{2\pi i \nu t}\, dt \right|^2$$

It includes all the anharmonicity and has no imaginary frequencies. The
Langevin friction $\gamma$ broadens every peak by $\gamma/2\pi$ = 1.6 THz.

In [ ]:
def md_dos(
    frames: list[Atoms], smearing: float = 0.15, max_frequency: float = 25.0
) -> tuple[np.ndarray, dict[str, np.ndarray]]:
    """Get the DOS of each element from the velocities, in states/THz per Li3PS4.

    The DOS is smeared with a Gaussian of width smearing in THz, mirrored at 0.
    It integrates to 3 states per atom.
    """
    velocities = np.array([atoms.get_velocities() for atoms in frames])
    masses = frames[0].get_masses()
    symbols = np.array(frames[0].get_chemical_symbols())
    nu = np.fft.rfftfreq(len(velocities), TIME_STEP)
    keep = nu <= max_frequency
    power = (np.abs(np.fft.rfft(velocities, axis=0)) ** 2).sum(2)[keep] * masses
    step = nu[1]
    half = int(4 * smearing / step)
    kernel = np.exp(-0.5 * (np.arange(-half, half + 1) * step / smearing) ** 2)
    n_formula_units = len(symbols) / 8
    out = {}
    for element in ("Li", "P", "S"):
        g = power[:, symbols == element].sum(1)
        g = np.convolve(np.concatenate([g[half:0:-1], g]), kernel, "same")[half:]
        n_states = 3 * (symbols == element).sum() / n_formula_units
        out[element] = g * n_states / (g.sum() * step)
    return nu[keep], out


nu_600k, dos_600k = md_dos(frames_600k)

In [ ]:
# vibrational DOS of the NVT MD in the relaxed 0 K cell, against the DOS of
# the force constants fitted to the same MD, at 300, 500 and 700 K: the fraction
# of the Li DOS below 5 THz, and the frequency in THz of the P-S stretch, the
# highest peak of the P DOS
VDOS_TEMPERATURES = (300, 500, 700)
LI_BELOW_5_THZ = {
    "OMAT-0-medium": {"MD": (0.15, 0.18, 0.23), "fit": (0.18, 0.27, 0.9)},
    "MATPES-PBE-0": {"MD": (0.21, 0.23, 0.25), "fit": (0.53, 0.98, 1.0)},
    "MATPES-r2SCAN-0": {"MD": (0.14, 0.17, 0.21), "fit": (0.19, 0.41, 0.99)},
}
PS_STRETCH = {
    "OMAT-0-medium": {"MD": (16.4, 16.0, 16.4), "fit": (15.9, 15.2, 12.9)},
    "MATPES-PBE-0": {"MD": (14.6, 15.0, 15.0), "fit": (13.8, 12.4, 12.0)},
    "MATPES-r2SCAN-0": {"MD": (16.3, 16.3, 16.3), "fit": (16.1, 15.2, 13.2)},
}

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for idx, model in enumerate(MODEL_FILES):
    for source, style in (("MD", "-o"), ("fit", "--s")):
        kwargs = {"color": f"C{idx}", "ms": 3, "label": f"{model}, {source}"}
        axes[0].plot(VDOS_TEMPERATURES, LI_BELOW_5_THZ[model][source], style, **kwargs)
        axes[1].plot(VDOS_TEMPERATURES, PS_STRETCH[model][source], style, **kwargs)
axes[0].set_ylabel("Fraction of the Li DOS below 5 THz")
axes[1].set_ylabel("P-S stretch (THz)")
for ax in axes:
    ax.set_xlabel("Temperature (K)")
axes[0].legend(fontsize=7)
fig.tight_layout()
plt.show()

![Li DOS below 5 THz and P-S stretch, MD against fit](finite_temperature_phonons_figures/s5_vdos.png)

Li DOS below 5 THz and P-S stretch, in the relaxed 0 K cell:

| Potential | T (K) | Li DOS below 5 THz, MD | fit | P-S stretch (THz), MD | fit |
|---|---|---|---|---|---|
| OMAT-0-medium | 300 | 15% | 18% | 16.4 | 15.9 |
| OMAT-0-medium | 500 | 18% | 27% | 16.0 | 15.2 |
| OMAT-0-medium | 700 | 23% | 90% | 16.4 | 12.9 |
| MATPES-PBE-0 | 300 | 21% | 53% | 14.6 | 13.8 |
| MATPES-PBE-0 | 500 | 23% | 98% | 15.0 | 12.4 |
| MATPES-PBE-0 | 700 | 25% | 100% | 15.0 | 12.0 |
| MATPES-r2SCAN-0 | 300 | 14% | 19% | 16.3 | 16.1 |
| MATPES-r2SCAN-0 | 500 | 17% | 41% | 16.3 | 15.2 |
| MATPES-r2SCAN-0 | 700 | 21% | 99% | 16.3 | 13.2 |

- At 300 K the MD and the fit agree, except with MACE-MATPES-PBE-0.
- At 700 K the fit puts 90 to 100% of the Li DOS below 5 THz. The MD puts 21 to
  25% there.
- The P-S stretch stays at 15 to 16.4 THz in the MD, but drops to 12 to 13 THz in
  the fit. The framework does not soften. The fit around fixed sites makes it
  look soft.

## Free energy from the MD

1. **F at 100 K by Frenkel-Ladd.** We switch from an Einstein crystal, with
   springs at the mean positions of the 100 K MD, to the MACE potential at 8
   values of $\lambda$. No Li hops at 100 K.
   $$F = F_E + \int_0^1 \langle U_\mathrm{MLIP} - U_E \rangle_\lambda\, d\lambda$$
2. **F at higher temperatures by Gibbs-Helmholtz,** with $U$ the mean energy of
   the MD, or the mean enthalpy for NPT. We stop at 800 K, where P and S start to
   diffuse.
   $$\frac{d(F/T)}{dT} = -\frac{U}{T^2}$$
3. **A quantum correction** from the MD DOS, since the MD is classical.

In [ ]:
from ase import units
from ase.calculators.calculator import Calculator, all_changes
from ase.constraints import FixCom
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from scipy import constants


class Switched(Calculator):
    """Energy and forces of lam U_MLIP + (1 - lam) U_E.

    delta holds U_MLIP - U_E of the last call.
    """

    implemented_properties = ("energy", "forces")

    def __init__(
        self,
        mlip: Calculator,
        positions: np.ndarray,
        springs: np.ndarray,
        lam: float,
    ) -> None:
        super().__init__()
        self.mlip, self.r0, self.k, self.lam = mlip, positions, springs, lam

    def calculate(
        self,
        atoms: Atoms | None = None,
        properties: tuple[str, ...] = ("energy",),
        system_changes: list[str] = all_changes,
    ) -> None:
        """Compute the switched energy and forces."""
        super().calculate(atoms, properties, system_changes)
        self.mlip.calculate(self.atoms, ["energy", "forces"], system_changes)
        d = self.atoms.positions - self.r0
        e_einstein = 0.5 * (self.k * (d**2).sum(1)).sum()
        self.delta = self.mlip.results["energy"] - e_einstein
        self.results = {
            "energy": self.lam * self.mlip.results["energy"]
            + (1 - self.lam) * e_einstein,
            "forces": self.lam * self.mlip.results["forces"]
            - (1 - self.lam) * self.k[:, None] * d,
        }


def einstein_free_energy(
    masses: np.ndarray, springs: np.ndarray, volume: float, temperature: float
) -> float:
    """Get the classical F (eV) of the Einstein crystal with a fixed center of mass.

    It includes the term for a center of mass that is free in the volume of the
    cell.
    """
    kt = units.kB * temperature
    hbar = constants.hbar / constants.e * units.second  # eV in ASE time units
    free = 3 * kt * np.log(hbar * np.sqrt(springs / masses) / kt).sum()
    variance = ((masses / masses.sum()) ** 2 * kt / springs).sum()
    return free + kt * np.log((2 * np.pi * variance) ** 1.5 / volume)


def frenkel_ladd(
    frames: list[Atoms],
    mlip: Calculator,
    temperature: float = 100.0,
    n_lambda: int = 8,
    steps: tuple[int, int] = (2000, 8000),
    seed: int = 103,
) -> float:
    """Get F (eV per atom) of the potential at the temperature of the frames.

    steps gives the equilibration and production steps of the MD at each
    lambda.
    """
    rng = np.random.default_rng(seed)
    atoms = frames[0].copy()
    masses = atoms.get_masses()
    pos = unwrapped_positions(frames)
    com = (pos * masses[None, :, None]).sum(1) / masses.sum()
    pos = pos - com[:, None] + com[0]
    atoms.positions = pos.mean(0)
    msd = ((pos - atoms.positions) ** 2).sum(2).mean(0)
    symbols = np.array(atoms.get_chemical_symbols())
    springs = np.empty(len(atoms))
    for element in set(symbols):
        springs[symbols == element] = (
            3 * units.kB * temperature / msd[symbols == element].mean()
        )
    nodes, weights = np.polynomial.legendre.leggauss(n_lambda)
    integral = 0.0
    for node, weight in zip(nodes, weights, strict=True):
        run = atoms.copy()
        run.set_constraint(FixCom())
        run.calc = Switched(mlip, atoms.positions.copy(), springs, (node + 1) / 2)
        MaxwellBoltzmannDistribution(run, temperature_K=temperature, rng=rng)
        Stationary(run)
        dyn = Langevin(
            run,
            units.fs,
            temperature_K=temperature,
            friction=0.01 / units.fs,
            fixcm=False,
            rng=rng,
        )
        dyn.run(steps[0])
        deltas = []
        for _ in range(steps[1] // 10):
            dyn.run(10)
            deltas.append(run.calc.delta)
        integral += weight / 2 * np.mean(deltas)
    f_e = einstein_free_energy(masses, springs, atoms.get_volume(), temperature)
    return (f_e + integral) / len(atoms)

On a Cu-Ag crystal with EMT at 10 K, these functions gave 0.247542 ± 0.000087 eV
against the exact 0.247599 eV. The next cell runs them on the 100 K run of
Section 4. It is off by default.

In [ ]:
RUN_FL = False
if RUN_FL:
    mlip = mace_mp(**calc_kwargs("OMAT-0-medium"))
    frames = nvt_trajectory("runs/Li3PS4_OMAT-0-medium_100K_nvt")
    f_100 = frenkel_ladd(frames, mlip)  # eV per atom

In [ ]:
PLANCK = constants.h / constants.e * 1e12  # eV/THz


def gibbs_helmholtz(
    t0: float, f0: float, temperatures: list[float], energies: list[float]
) -> np.ndarray:
    """Get F at each temperature from F at t0 and the mean energy U.

    F and U are in eV per atom. temperatures starts at t0, and U is linear in T
    between the temperatures.
    """
    f_over_t = [f0 / t0]
    for i in range(1, len(temperatures)):
        t1, t2 = temperatures[i - 1], temperatures[i]
        b = (energies[i] - energies[i - 1]) / (t2 - t1)
        a = energies[i - 1] - b * t1
        f_over_t.append(f_over_t[-1] - (a * (1 / t1 - 1 / t2) + b * np.log(t2 / t1)))
    return np.array(f_over_t) * np.array(temperatures)


def quantum_correction(
    nu: np.ndarray, dos: np.ndarray, temperature: float
) -> tuple[float, float]:
    """Get the quantum minus the classical harmonic F (eV) and S (eV/K) of a DOS.

    nu is in THz. The result is per formula unit for a DOS per formula unit, as
    md_dos gives it. Divide by 8 for a value per atom.
    """
    keep = nu > 0
    nu, g, step = nu[keep], dos[keep], nu[1] - nu[0]
    kt = units.kB * temperature
    x = PLANCK * nu / kt
    df = PLANCK * nu / 2 + kt * (np.log1p(-np.exp(-x)) - np.log(x))
    ds = units.kB * (x / np.expm1(x) - np.log1p(-np.exp(-x)) - 1 + np.log(x))
    return (df * g).sum() * step, (ds * g).sum() * step

In [ ]:
# per formula unit, at 100 to 800 K: F in kJ/mol relative to the relaxed 0 K
# energy and S in J/(K mol). From the MD in the relaxed 0 K cell (nvt) and in the
# NPT cell (npt), from the force constants fitted in the 0 K cell with U0 added,
# and from the 0 K force constants.
THERMO_T = (100, 200, 300, 400, 500, 600, 700, 800)
MD_F = {
    "nvt": {
        "OMAT-0-medium": (31.5, 21.2, 2.7, -22.9, -52.3, -87.4, -125.5, -167.4),
        "MATPES-PBE-0": (27.2, 13.7, -7.6, -34.6, -67.0, -103.4, -143.5, -186.4),
        "MATPES-r2SCAN-0": (31.2, 19.2, 0.3, -25.0, -55.6, -90.5, -129.0, -170.8),
    },
    "npt": {
        "OMAT-0-medium": (31.1, 19.6, -0.4, -26.5, -58.7, -94.0, -134.6, -178.8),
        "MATPES-PBE-0": (27.6, 13.6, -9.0, -36.5, -69.8, -106.9, -148.2, -192.3),
        "MATPES-r2SCAN-0": (30.9, 18.5, -1.4, -27.5, -58.6, -94.8, -134.6, -178.4),
    },
}
MD_S = {
    "nvt": {
        "OMAT-0-medium": (63, 150, 218, 273, 317, 363, 401, 435),
        "MATPES-PBE-0": (79, 169, 242, 296, 343, 384, 415, 441),
        "MATPES-r2SCAN-0": (68, 154, 221, 278, 326, 361, 400, 433),
    },
    "npt": {
        "OMAT-0-medium": (68, 158, 230, 286, 334, 380, 419, 461),
        "MATPES-PBE-0": (87, 178, 246, 306, 351, 392, 426, 456),
        "MATPES-r2SCAN-0": (76, 159, 229, 284, 334, 379, 417, 449),
    },
}
FIT_F = {
    "OMAT-0-medium": (32.2, 19.7, 0.3, -36.5, -60.3, -118.8, -166.6, -219.7),
    "MATPES-PBE-0": (27.1, 12.4, -17.5, -49.0, -95.1, -139.5, -184.4, -240.1),
    "MATPES-r2SCAN-0": (31.3, 18.9, -0.7, -29.4, -68.5, -115.1, -168.9, -225.2),
}
FIT_S = {
    "OMAT-0-medium": (68, 155, 223, 305, 333, 410, 452, 493),
    "MATPES-PBE-0": (82, 179, 273, 326, 391, 433, 457, 495),
    "MATPES-r2SCAN-0": (65, 155, 224, 288, 350, 401, 449, 489),
}
HARMONIC_F = {
    "OMAT-0-medium": (35.4, 25.5, 8.8, -13.4, -40.1, -70.3, -103.7, -139.7),
    "MATPES-PBE-0": (30.0, 18.0, -1.4, -26.7, -56.5, -90.1, -126.8, -166.2),
    "MATPES-r2SCAN-0": (35.6, 26.0, 9.6, -12.3, -38.4, -68.2, -101.0, -136.5),
}
HARMONIC_S = {
    "OMAT-0-medium": (59, 135, 197, 246, 286, 319, 348, 373),
    "MATPES-PBE-0": (75, 160, 226, 277, 318, 352, 381, 407),
    "MATPES-r2SCAN-0": (58, 132, 193, 241, 281, 314, 342, 367),
}

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True)
for col, model in enumerate(MODEL_FILES):
    for row, (md, fit, harmonic) in enumerate(
        ((MD_F, FIT_F, HARMONIC_F), (MD_S, FIT_S, HARMONIC_S))
    ):
        ax = axes[row, col]
        ax.plot(THERMO_T, md["nvt"][model], "-o", ms=3, label="MD, 0 K cell")
        ax.plot(THERMO_T, md["npt"][model], "-s", ms=3, label="MD, NPT cell")
        ax.plot(THERMO_T, fit[model], "--", label="fit")
        ax.plot(
            THERMO_T, harmonic[model], ":", color="gray", label="0 K force constants"
        )
    axes[0, col].set_title(model)
    axes[1, col].set_xlabel("Temperature (K)")
axes[0, 0].set_ylabel("F (kJ/mol)")
axes[1, 0].set_ylabel("S (J/(K mol))")
axes[1, 0].legend(fontsize=8)
fig.tight_layout()
plt.show()

![Free energy and entropy of Li3PS4](finite_temperature_phonons_figures/s5_thermo.png)

S at 700 K in J/(K mol) per formula unit:

| Potential | MD, 0 K cell | MD, NPT cell | fit | 0 K force constants |
|---|---|---|---|---|
| OMAT-0-medium | 401 | 419 | 452 | 348 |
| MATPES-PBE-0 | 415 | 426 | 457 | 381 |
| MATPES-r2SCAN-0 | 400 | 417 | 449 | 342 |

The fitted F includes U₀, the mean potential energy of its MD minus 3kT/2 per
atom.

- Up to 300 K the fit and the MD agree within 1 to 2.5 kJ/mol, except with
  MACE-MATPES-PBE-0.
- Where Li hops, the fit overestimates S at 700 K by 40 to 50 J/(K mol).
- The 0 K force constants underestimate it by 35 to 60 J/(K mol). So does the
  entropy in the Materials Project's Harmonic Phonon Database.

## γ against β

The phase with the lowest G is stable, so a crossing of G(β) and G(γ) is a
transition. The Materials Project has no γ entry and no α framework. We take γ
from Kaup et al. (Table S1), which matches Homma et al. (COD 1570309).

In [ ]:
from pymatgen.core import Lattice

gamma = Structure.from_spacegroup(
    "Pmn2_1",
    Lattice.orthorhombic(7.7557, 6.5627, 6.1362),
    ["Li", "Li", "P", "S", "S", "S"],
    [
        [0.2441, 0.3122, -0.0029],
        [0, 0.1493, 0.475],
        [0, 0.8174, 0.9955],
        [0.2177, 0.6717, 0.8864],
        [0, 0.1115, 0.8935],
        [0, 0.8094, 0.3274],
    ],
)
GAMMA_SUPERCELL = (2, 2, 2)  # 128 atoms
gamma.composition, gamma.get_space_group_info()

At 0 K β lies above γ by 4.7 meV/atom with MACE-OMAT-0-medium, 11.9 with
MACE-MATPES-PBE-0 and 10.5 with MACE-MATPES-r2SCAN-0. We run γ like β, with the
NPT MD and Frenkel-Ladd. The loop is off by default.

In [ ]:
RUN_GAMMA = False
gamma_docs = {}
if RUN_GAMMA:
    for model in MODEL_FILES:
        for temperature in LI3PS4_TEMPERATURES:
            gamma_docs[model, temperature] = run_ft(
                gamma,
                GAMMA_SUPERCELL,
                model,
                f"runs/gamma_{model}_{temperature}K_npt",
                temperature,
                relax=True,
                run_npt=True,
            )

In [ ]:
# G(beta) - G(gamma) in meV/atom and S of each phase in J/(K mol) per formula
# unit, at 100 to 800 K and zero pressure, with the quantum correction
PHASE_T = (100, 200, 300, 400, 500, 600, 700, 800)
DELTA_G = {
    "OMAT-0-medium": (-0.3, -1.8, -3.6, -5.1, -6.6, -7.3, -6.9, -4.9),
    "MATPES-PBE-0": (7.1, 5.5, 3.4, 1.1, -1.1, -2.2, -1.9, -0.4),
    "MATPES-r2SCAN-0": (3.0, 1.6, -0.5, -2.4, -4.6, -7.2, -9.5, -10.5),
}
PHASE_S = {
    "gamma": {
        "OMAT-0-medium": (80, 156, 222, 278, 327, 380, 430, 484),
        "MATPES-PBE-0": (92, 172, 238, 292, 340, 391, 436, 474),
        "MATPES-r2SCAN-0": (79, 154, 219, 273, 319, 360, 403, 450),
    },
    "beta": {
        "OMAT-0-medium": (90, 168, 235, 290, 338, 383, 422, 463),
        "MATPES-PBE-0": (103, 188, 255, 311, 355, 395, 429, 458),
        "MATPES-r2SCAN-0": (91, 169, 235, 288, 337, 382, 419, 451),
    },
}

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax in axes:
    ax.axvspan(523, 573, color="0.9")
axes[0].axhline(0, color="gray", lw=0.5)
for idx, model in enumerate(MODEL_FILES):
    axes[0].plot(PHASE_T, DELTA_G[model], "-o", color=f"C{idx}", ms=3, label=model)
    for phase, style in (("gamma", "--"), ("beta", "-")):
        axes[1].plot(PHASE_T, PHASE_S[phase][model], style, color=f"C{idx}")
axes[0].set_ylabel("G(beta) - G(gamma) (meV/atom)")
axes[1].set_ylabel("S (J/(K mol)), beta solid, gamma dashed")
for ax in axes:
    ax.set_xlabel("Temperature (K)")
axes[0].legend(fontsize=8)
fig.tight_layout()
plt.show()

![G(beta) - G(gamma) and entropies](finite_temperature_phonons_figures/s5_phases.png)

G(β) - G(γ) in meV/atom:

| Potential | 100 K | 200 K | 300 K | 400 K | 500 K | 600 K | 700 K | 800 K |
|---|---|---|---|---|---|---|---|---|
| OMAT-0-medium | -0.3 | -1.8 | -3.6 | -5.1 | -6.6 | -7.3 | -6.9 | -4.9 |
| MATPES-PBE-0 | 7.1 | 5.5 | 3.4 | 1.1 | -1.1 | -2.2 | -1.9 | -0.4 |
| MATPES-r2SCAN-0 | 3.0 | 1.6 | -0.5 | -2.4 | -4.6 | -7.2 | -9.5 | -10.5 |

β becomes stable above 278 K with MACE-MATPES-r2SCAN-0 and above 450 K with
MACE-MATPES-PBE-0, against 523 to 573 K in experiment. With MACE-OMAT-0-medium β
is already lower at 100 K. The second crossing of MACE-MATPES-PBE-0 at 822 K
does not count, since γ flows there.

- β is softer. At 100 K its classical F lies only 0.2 to 7.9 meV/atom above γ,
  much less than the gap at 0 K.
- The quantum correction is about 5 meV/atom for each phase at 500 K, but it
  changes G(β) - G(γ) by 1.1 meV/atom or less.
- The error of each Frenkel-Ladd F is about 0.1 meV/atom.

## What this approach cannot do

- **α.** Its ordered versions relax into distorted structures 18 to 22 meV/atom
  above γ, so Frenkel-Ladd has no stable start. α needs a treatment of its Li
  disorder, for example a cluster expansion.
- **Melting.** It needs a reference for the liquid.
- **Kinetics.** The free energies give the equilibrium temperature only, not the
  hysteresis seen in experiment.

## Why the γ to β temperature is off

- **The potentials.** 1 meV/atom in G(β) - G(γ) moves the crossing by about
  50 K, and the 0 K gaps differ by 7 meV/atom between the potentials. Kam et al.
  found r²SCAN DFT 200 to 300 K too low (arXiv:2307.00878), and
  MACE-MATPES-r2SCAN-0 falls in the same range.
- **One ordering of β.** Below about 500 K the MD misses part of the Li disorder
  of β.
- **Size and length.** 128 atoms and 8 ps of MD.

## Known limitations

- Without `run_npt=True` the MD keeps the volume of the starting structure. The
  NPT cell comes from one MD run and carries its noise.
- MACE gives no Born charges, so no non-analytical correction is applied. A
  `born_maker` adds it, either `ForceFieldDielectricMaker` with MACE-Field or a
  VASP `DielectricMaker`.
- The trajectory check uses fixed limits. The Lindemann limit of 0.15 is that of
  fcc solids, and light atoms that hop can pass it while the crystal stays solid.
- Each result comes from one trajectory. Runs differ by up to 0.1 THz on average
  and 0.3 THz in the highest frequency (Section 3).